# Chapter 1 — Tensors as 1D Arrays

> Course: **llm.c — Zero to Hero**, Chapter 1 of ~20.
> Prerequisites: Python, basic PyTorch, comfort with neural-network math. **No prior C/C++ experience required.**

Welcome! Before we can understand how `llm.c` trains a GPT-2 model in pure C, we need to understand how a tensor — the fundamental data structure of deep learning — actually lives in memory.

In PyTorch, tensors feel like multi-dimensional objects you can slice, reshape, and index naturally:

```python
x = torch.zeros(2, 3, 4)
x[1, 2, 3] = 42.0
```

Under the hood, however, a tensor is just **a pointer to a contiguous block of memory + some shape metadata**. PyTorch hides all the indexing arithmetic for you. In C, *you* are the indexing arithmetic.

### Learning objectives

By the end of this chapter you will be able to:

- Allocate and free a tensor in C with `malloc` and `free`.
- Translate `(b, t, c)` indexing into a single flat 1D offset by hand.
- Read and write the most common tensor-pointer pattern in `llm.c`:
  ```c
  float* out_bt = out + b*T*C + t*C;
  ```
- Walk through how the **encoder** forward pass in `train_gpt2.c` stores its outputs.


## 1. The Concept — Memory is Always 1D

RAM (and GPU memory) is a flat 1D array of bytes. Anything multi-dimensional is a *convention* layered on top of that flat array.

PyTorch's `torch.zeros(B, T, C)` allocates `B*T*C*4` bytes (for float32) and stores them in **row-major order** (also called *C order*, the default in both PyTorch and NumPy):

```
buffer index:  0          1          ...   C-1        C          C+1        ...
content:       x[0,0,0]   x[0,0,1]   ...   x[0,0,C-1] x[0,1,0]   x[0,1,1]   ...
```

The element at logical position `(b, t, c)` lives at buffer offset:

$$\text{flat\_index}(b,t,c) = b\cdot(T\cdot C) + t\cdot C + c = b\cdot s_B + t\cdot s_T + c\cdot s_C$$

where the **strides** $s_B, s_T, s_C$ tell you how many elements to skip to advance one step along each axis. For a contiguous `(B, T, C)` tensor:

| axis | stride (in elements) |
|------|----------------------|
| `B`  | `T * C`              |
| `T`  | `C`                  |
| `C`  | `1`                  |

Every tensor operation in `llm.c` boils down to *computing this offset and reading or writing through it.*


## 2. PyTorch Baseline

Let's first confirm this picture in PyTorch, where we can see strides and the flat buffer directly.


In [ ]:
import torch

B, T, C = 2, 3, 4  # batch, time, channels — typical GPT-2 activation shape
x = torch.arange(B*T*C, dtype=torch.float32).reshape(B, T, C)

print("Shape:        ", tuple(x.shape))
print("Strides (el): ", x.stride())
print("Contiguous?   ", x.is_contiguous())
print("Flat 1D buffer:")
print(x.flatten().tolist())

b, t, c = 1, 2, 3
flat_idx = b * (T * C) + t * C + c
print(f"\nx[{b},{t},{c}]                  = {x[b,t,c].item()}")
print(f"flat[{b}*T*C + {t}*C + {c} = {flat_idx}] = {x.flatten()[flat_idx].item()}")


You should see that `x[1,2,3]` and `flat[23]` give the same value. PyTorch did the math `1*12 + 2*4 + 3 = 23` for you. In C, **you** will write that math.


## 3. The C Way

In C, there are no `Tensor` objects. A tensor is just:

```c
float* x = (float*) malloc(B * T * C * sizeof(float));
```

That's it. You get back `x`, a pointer to the first byte of `B*T*C*4` bytes of raw memory. There is no `.shape`. The shape lives **only in your head and in the variables you pass alongside the pointer** (`B`, `T`, `C`).

When you're done, you must release the memory yourself:

```c
free(x);
```

If you forget `free`, you have a memory leak. If you `free` twice, you crash. Welcome to manual memory management.

### How `llm.c` reads from a `(B, T, C)` activation

Open [`train_gpt2.c`](train_gpt2.c) and look at `encoder_forward` (lines 35–55). The relevant pattern is:

```c
void encoder_forward(float* out,
                     int* inp, float* wte, float* wpe,
                     int B, int T, int C) {
    // out is (B,T,C). At each position (b,t), a C-dimensional vector.
    // inp is (B,T) of integers (token ids)
    // wte is (V,C) of token embeddings, wpe is (maxT,C) of position embeddings
    for (int b = 0; b < B; b++) {
        for (int t = 0; t < T; t++) {
            // seek to the output position in out[b,t,:]
            float* out_bt = out + b * T * C + t * C;
            // get the index of the token at inp[b, t]
            int ix = inp[b * T + t];
            // seek to the position in wte corresponding to the token
            float* wte_ix = wte + ix * C;
            // seek to the position in wpe corresponding to the position
            float* wpe_t = wpe + t * C;
            // add the two vectors and store the result in out[b,t,:]
            for (int i = 0; i < C; i++) {
                out_bt[i] = wte_ix[i] + wpe_t[i];
            }
        }
    }
}
```

The five lines that matter for *this* chapter:

```c
float* out_bt = out + b * T * C + t * C;   // pointer to out[b, t, :]
int ix = inp[b * T + t];                    // scalar at inp[b, t]
float* wte_ix = wte + ix * C;               // pointer to wte[ix, :]
float* wpe_t = wpe + t * C;                 // pointer to wpe[t, :]
out_bt[i] = wte_ix[i] + wpe_t[i];           // pointwise: out[b,t,i] = wte[ix,i] + wpe[t,i]
```

Note the trick: instead of indexing the full 1D buffer every time, `llm.c` computes a **pointer to a row** once (`out_bt`), then uses `out_bt[i]` to read elements *within that row*. This is the C equivalent of PyTorch's `out[b, t]` returning a 1D view — except in C, a "view" *is* literally just a pointer.


## 4. The Translation Bridge

| PyTorch | C in `llm.c` |
|---|---|
| `x = torch.zeros(B, T, C)` | `float* x = (float*) calloc(B*T*C, sizeof(float));` |
| `del x` (or scope exit) | `free(x);` |
| `x.shape` | A separate `int B, T, C;` you carry around |
| `x.stride()` | You compute it: `T*C, C, 1` |
| `x[b, t]` (a row view) | `float* x_bt = x + b*T*C + t*C;` |
| `x[b, t, c]` (scalar) | `x[b*T*C + t*C + c]` *or* `x_bt[c]` |
| `x[b, t] = y` (slice assign) | `for (int i=0;i<C;i++) x_bt[i] = y[i];` |
| `x.is_contiguous()` | Always true in `llm.c`. There are no non-contiguous tensors here. |
| Bounds checking on `x[b,t,c]` | None. Read past the end and you get garbage (or a crash). |

The mental model that makes this all click:

> **A `float*` is a tensor whose shape lives in your variables.**

Once you internalize this, every line of [`train_gpt2.c`](train_gpt2.c) becomes readable.


## 5. Toy Example — Build and Index a Tensor in C

Time to actually run some C from this notebook. We'll use Jupyter's `%%writefile` magic to dump source to disk, then `gcc` to compile it, then run the binary.

> **Note:** Make sure you have `gcc` installed. On Linux: `sudo apt install build-essential`. On macOS: `xcode-select --install`.


In [ ]:
# Make a folder for our chapter scratch files
!mkdir -p course/ch01_build


In [ ]:
%%writefile course/ch01_build/hello_world.c
#include <stdio.h>

int main() {
    printf("Hello, World!\n");
    return 0;
}

In [ ]:
!gcc -O2 course/ch01_build/hello_world.c -o course/ch01_build/hello_world && ./course/ch01_build/hello_world

In [ ]:
%%writefile course/ch01_build/tensor_basics.c
#include <stdio.h>
#include <stdlib.h>

int main(void) {
    const int B = 2, T = 3, C = 4;

    // Allocate a flat (B*T*C) buffer. Casting malloc's void* to float* is the C idiom.
    float* x = (float*) malloc(B * T * C * sizeof(float));
    if (x == NULL) { fprintf(stderr, "malloc failed\n"); return 1; }

    // Fill it: x[b, t, c] = 100*b + 10*t + c
    for (int b = 0; b < B; b++) {
        for (int t = 0; t < T; t++) {
            for (int c = 0; c < C; c++) {
                x[b*T*C + t*C + c] = 100.0f*b + 10.0f*t + (float)c;
            }
        }
    }

    // Print the underlying flat buffer
    printf("Flat 1D view (length %d):\n", B*T*C);
    for (int i = 0; i < B*T*C; i++) printf("%6.0f ", x[i]);
    printf("\n\n");

    // Use the row-pointer trick to print x[1, 2, :]
    int b = 1, t = 2;
    float* x_bt = x + b*T*C + t*C;
    printf("x[%d, %d, :] = ", b, t);
    for (int i = 0; i < C; i++) printf("%6.0f ", x_bt[i]);
    printf("\n");

    free(x);
    return 0;
}


In [ ]:
!gcc -O2 course/ch01_build/tensor_basics.c -o course/ch01_build/tensor_basics && ./course/ch01_build/tensor_basics


You should see the 1D buffer (24 values) followed by `x[1, 2, :] = 120 121 122 123` — confirming the offset math `1*12 + 2*4 = 20`, plus `100 + 20 + c`.


## 6. Pointer Arithmetic Deep Dive

The line that confuses every C beginner:

```c
float* out_bt = out + b * T * C + t * C;
```

Read this as: *"Take pointer `out`, advance it by `b*T*C + t*C` **float-sized slots**, and call the result `out_bt`."*

Three things you must know:

1. **Pointer arithmetic is element-wise, not byte-wise.**
   `out + 1` advances by `sizeof(float) = 4` bytes, *not* by 1 byte. The compiler scales by `sizeof(*out)` for you.
2. **`p[i]` is just `*(p + i)`.**
   The square-bracket syntax is sugar for *"advance pointer by `i`, then dereference"*. This is why `out_bt[i]` works.
3. **A pointer plus an integer is still a pointer.**
   It carries no shape, no length. If `out` was allocated for `B*T*C` floats and you do `out + 99999`, the compiler is happy. The CPU is happy. Your program may crash later. **This is C.**


## 7. TODO Exercise 1 — Compute Tensor Offsets

Below is a short program that allocates a `(B, T, C)` tensor, fills it with `x[b,t,c] = 100*b + 10*t + c`, and is supposed to print three specific elements. **The flat-offset lines are missing — fill them in.**

Edit the cell, run it (which writes the file), then run the compile/run cell below it.


In [ ]:
%%writefile course/ch01_build/exercise1.c
#include <stdio.h>
#include <stdlib.h>

int main(void) {
    const int B = 2, T = 3, C = 4;
    float* x = (float*) malloc(B*T*C*sizeof(float));
    for (int b = 0; b < B; b++)
        for (int t = 0; t < T; t++)
            for (int c = 0; c < C; c++)
                x[b*T*C + t*C + c] = 100.0f*b + 10.0f*t + (float)c;

    // TODO: compute the flat index for x[0, 1, 2] and read x at that index
    int idx_a = 0 * T * C + 1 * C + 2; // <-- replace 0 with the right expression in terms of B, T, C
    printf("x[0,1,2] = %.0f  (expected 12)\n", x[idx_a]);

    // TODO: same for x[1, 0, 3]
    int idx_b = 1 * T * C + 0 * C + 3;
    printf("x[1,0,3] = %.0f  (expected 103)\n", x[idx_b]);

    // TODO: get a row-pointer to x[1, 2, :] using the "x + b*T*C + t*C" idiom
    float* row = x + 1 * T * C + 2 * C;  // <-- replace NULL with the right pointer expression
    printf("x[1,2,0] = %.0f  (expected 120)\n", row[0]);

    free(x);
    return 0;
}


In [ ]:
!gcc -O2 course/ch01_build/exercise1.c -o course/ch01_build/exercise1 && ./course/ch01_build/exercise1


Expected output once you've filled in the TODOs:

```
x[0,1,2] = 12  (expected 12)
x[1,0,3] = 103  (expected 103)
x[1,2,0] = 120  (expected 120)
```

If you got `0` for any of them, your offset is wrong. Try again before peeking below.


### Solution to Exercise 1
*(don't peek until you've tried — answers in the next two cells)*


In [ ]:
%%writefile course/ch01_build/exercise1_solution.c
#include <stdio.h>
#include <stdlib.h>

int main(void) {
    const int B = 2, T = 3, C = 4;
    float* x = (float*) malloc(B*T*C*sizeof(float));
    for (int b = 0; b < B; b++)
        for (int t = 0; t < T; t++)
            for (int c = 0; c < C; c++)
                x[b*T*C + t*C + c] = 100.0f*b + 10.0f*t + (float)c;

    int idx_a = 0*T*C + 1*C + 2;     // = 6
    printf("x[0,1,2] = %.0f  (expected 12)\n", x[idx_a]);

    int idx_b = 1*T*C + 0*C + 3;     // = 15
    printf("x[1,0,3] = %.0f  (expected 103)\n", x[idx_b]);

    float* row = x + 1*T*C + 2*C;    // pointer to x[1, 2, :]
    printf("x[1,2,0] = %.0f  (expected 120)\n", row[0]);

    free(x);
    return 0;
}


In [ ]:
!gcc -O2 course/ch01_build/exercise1_solution.c -o course/ch01_build/exercise1_solution && ./course/ch01_build/exercise1_solution


## 8. TODO Exercise 2 — Reproduce a Slice of `encoder_forward`

Now let's do something useful: reproduce the **inner loop of `encoder_forward`** from `train_gpt2.c` for a single `(b, t)` position.

You're given:

- `wte` of shape `(V, C)` — row `i` is the embedding of token id `i`
- `wpe` of shape `(maxT, C)` — row `t` is the embedding of sequence position `t`
- a token id `ix` and a position `t`

Your job: write `out[i] = wte[ix, i] + wpe[t, i]` for every `i` in `[0, C)`. This is *exactly* what GPT-2 does in the very first layer of its forward pass, every single time you call `model(tokens)`.


In [ ]:
%%writefile course/ch01_build/exercise2.c
#include <stdio.h>
#include <stdlib.h>

int main(void) {
    const int V = 5, maxT = 4, C = 3;
    // Tiny embedding tables, hand-crafted so we can verify by eye
    float wte[5*3] = { 0,0,0,  10,20,30,  40,50,60,  70,80,90,  -1,-1,-1 };
    float wpe[4*3] = { 100,200,300,  400,500,600,  700,800,900,  1000,1000,1000 };

    int ix = 1;   // pretend the token at this position is token id 1
    int t  = 2;   // and we are at position 2 in the sequence

    float out[3];

    // TODO: pointer to wte[ix, :]   (hint: wte + ix * C)
    float* wte_ix = wte + ix * C;

    // TODO: pointer to wpe[t, :]
    float* wpe_t  = wpe + t * C;

    // TODO: write the elementwise sum into out
    for (int i = 0; i < C; i++) {
        out[i] = wte_ix[i] + wpe_t[t];  // <-- replace this with the elementwise sum
        out[i] = wte_ix[i] + wpe_t[i];
    }

    printf("out = [%.0f, %.0f, %.0f]  (expected [710, 820, 930])\n",
           out[0], out[1], out[2]);
    return 0;
}


In [ ]:
!gcc -O2 course/ch01_build/exercise2.c -o course/ch01_build/exercise2 && ./course/ch01_build/exercise2


Expected: `out = [710, 820, 930]`, which is `wte[1] + wpe[2] = [10,20,30] + [700,800,900]`.


### Solution to Exercise 2

In [ ]:
%%writefile course/ch01_build/exercise2_solution.c
#include <stdio.h>
#include <stdlib.h>

int main(void) {
    const int V = 5, maxT = 4, C = 3;
    float wte[5*3] = { 0,0,0,  10,20,30,  40,50,60,  70,80,90,  -1,-1,-1 };
    float wpe[4*3] = { 100,200,300,  400,500,600,  700,800,900,  1000,1000,1000 };

    int ix = 1, t = 2;
    float out[3];

    float* wte_ix = wte + ix * C;
    float* wpe_t  = wpe + t  * C;
    for (int i = 0; i < C; i++) {
        out[i] = wte_ix[i] + wpe_t[i];
    }

    printf("out = [%.0f, %.0f, %.0f]  (expected [710, 820, 930])\n",
           out[0], out[1], out[2]);
    return 0;
}


In [ ]:
!gcc -O2 course/ch01_build/exercise2_solution.c -o course/ch01_build/exercise2_solution && ./course/ch01_build/exercise2_solution


If this prints `[710, 820, 930]`, congratulations — **you have just hand-written the first three lines of GPT-2's forward pass.** Every subsequent layer in `train_gpt2.c` is built on the same row-pointer idiom you just used.


## Recap

You now know:

- Multi-dimensional tensors are stored as a flat 1D buffer in **row-major** order.
- `x[b, t, c]` translates to `x[b*T*C + t*C + c]`.
- `llm.c` favors the **row-pointer trick** `float* x_bt = x + b*T*C + t*C;` so that the inner loop reads `x_bt[i]` like a 1D vector.
- `malloc` and `free` are now *your* responsibility — there is no garbage collector.
- A `float*` is a tensor whose shape lives in your variables.

### What's next

**Chapter 2 — The Embedding Layer.** We'll go beyond a single position and walk through *all* of `encoder_forward` and its `encoder_backward` partner, including how gradients flow back into the token and position embedding tables. We'll also write our first OpenMP-parallelized loop.
